In [1]:
!pip install PyWavelets gdown --quiet
import os
import shutil
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


PyTorch version: 2.10.0+cu128
CUDA available: True


In [2]:
REPO_URL = "https://github.com/agamswami/SimpleTMG.git"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

!sed -i 's/np\.Inf/np.inf/g' utils/tools.py


Cloning into '/kaggle/working/SimpleTM'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 175 (delta 77), reused 174 (delta 76), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 18.32 MiB | 21.56 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Working directory: /kaggle/working/SimpleTM


In [3]:
FILE_ID = "1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    !gdown https://drive.google.com/uc?id={FILE_ID} -O {OUTPUT_PATH}
    import zipfile
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Dataset extracted successfully.")
else:
    print("Dataset already exists.")


Downloading...
From (original): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx
From (redirected): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx&confirm=t&uuid=2bc7d37a-3a1f-4c92-9576-0aa6e9c3d2bb
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 171M/171M [00:03<00:00, 49.0MB/s]
Dataset extracted successfully.


In [4]:
datasets = [
    {"name": "ETTh1", "data": "ETTh1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "ETTh2", "data": "ETTh2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "ETTm1", "data": "ETTm1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "ETTm2", "data": "ETTm2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "weather", "data": "custom", "root": "./dataset/SimpleTM_datasets/weather", "path": "weather.csv", "enc_in": 21, "d_model": 32, "d_ff": 32},
    # {"name": "electricity", "data": "custom", "root": "./dataset/SimpleTM_datasets/electricity", "path": "electricity.csv", "enc_in": 321, "d_model": 128, "d_ff": 128},
    # {"name": "traffic", "data": "custom", "root": "./dataset/SimpleTM_datasets/traffic", "path": "traffic.csv", "enc_in": 862, "d_model": 256, "d_ff": 256},
]

experiments = [
    {"tag": "SWT_original", "model": "SimpleTM_SWT", "attention_mode": "original"},
    {"tag": "SWT_dual", "model": "SimpleTM_SWT", "attention_mode": "dual"},
    {"tag": "FFT_original", "model": "SimpleTM_FFT", "attention_mode": "original"},
    {"tag": "FFT_dual", "model": "SimpleTM_FFT", "attention_mode": "dual"},
]

OUTPUT_DIR = "/kaggle/working/SimpleTM_DualAttention_Results"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
if os.path.exists("./checkpoints"):
    shutil.rmtree("./checkpoints")
if os.path.exists("result_long_term_forecast.txt"):
    os.remove("result_long_term_forecast.txt")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

print(f"Datasets: {len(datasets)}")
print(f"Experiment variants per dataset: {len(experiments)}")
print(f"Total runs planned: {len(datasets) * len(experiments)}")


Datasets: 5
Experiment variants per dataset: 4
Total runs planned: 20


In [5]:
import subprocess

for ds in datasets:
    print("\n" + "=" * 90)
    print(f"DATASET: {ds['name']} | variates={ds['enc_in']}")
    print("=" * 90 + "\n")

    for exp in experiments:
        print(f">>> Training {exp['tag']} on {ds['name']} <<<\n")

        unique_model_id = f"{ds['name']}_{exp['model']}_{exp['attention_mode']}"

        cmd = [
            "python", "-u", "run.py",
            "--is_training", "1",
            "--model", exp["model"],
            "--attention_mode", exp["attention_mode"],
            "--model_id", unique_model_id,
            "--data", ds["data"],
            "--root_path", ds["root"],
            "--data_path", ds["path"],
            "--features", "M",
            "--seq_len", "96",
            "--pred_len", "96",
            "--e_layers", "1",
            "--d_model", str(ds["d_model"]),
            "--d_ff", str(ds["d_ff"]),
            "--enc_in", str(ds["enc_in"]),
            "--dec_in", str(ds["enc_in"]),
            "--c_out", str(ds["enc_in"]),
            "--wv", "db1",
            "--m", "3",
            "--alpha", "1.0",
            "--learning_rate", "0.0001",
            "--batch_size", "32",
            "--train_epochs", "10",
            "--patience", "3",
            "--num_workers", "2",
            "--checkpoints", f"{OUTPUT_DIR}/checkpoints/",
            "--fix_seed", "2025"
        ]

        try:
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                print(line, end="")
            process.wait()

            if process.returncode == 0:
                print(f"\nSUCCESS: Finished {exp['tag']} on {ds['name']}\n")
            else:
                print(f"\nERROR: {exp['tag']} failed on {ds['name']} (exit code {process.returncode})\n")
        except Exception as e:
            print(f"\nFATAL: Exception running {exp['tag']} on {ds['name']}: {e}\n")

    print("-" * 90)
    print(f"Finished all variants on {ds['name']}")
    print("-" * 90)

print("\nGathering generated PDF plots...")
if os.path.exists('./checkpoints'):
    for root_dir, dirs, files in os.walk('./checkpoints'):
        for file in files:
            if file.endswith('.pdf'):
                folder_name = os.path.basename(root_dir)
                source_path = os.path.join(root_dir, file)
                target_plot_dir = os.path.join(OUTPUT_DIR, 'plots', folder_name)
                os.makedirs(target_plot_dir, exist_ok=True)
                shutil.copy(source_path, os.path.join(target_plot_dir, file))
    print(f"Copied PDF plots to {OUTPUT_DIR}/plots/")

if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt'))
    print("Copied metric text results to output directory.")



DATASET: ETTh1 | variates=7

>>> Training SWT_original on ETTh1 <<<

Args in experiment:
Namespace(is_training=1, model_id='ETTh1_SimpleTM_SWT_original', model='SimpleTM_SWT', data='ETTh1', root_path='./dataset/SimpleTM_datasets/ETT-small', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='/kaggle/working/SimpleTM_DualAttention_Results/checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False,

In [6]:
import pandas as pd
import re

results_file = os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt')

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        content = f.read()

    entries = content.strip().split('\n\n')
    results = []
    pattern = re.compile(r'^(?P<dataset>[^_]+)_(?P<model>SimpleTM_(?:SWT|FFT))_(?P<attention>original|dual)_')

    for entry in entries:
        lines = entry.strip().split('\n')
        if len(lines) < 2:
            continue

        setting = lines[0].strip()
        metrics_line = lines[-1].strip()
        match = pattern.search(setting)
        if not match:
            continue

        dataset_name = match.group('dataset')
        model_type = match.group('model')
        attention_mode = match.group('attention')
        tokenization = 'SWT' if model_type.endswith('SWT') else 'FFT'
        variant = f"{tokenization}_{attention_mode}"

        mse_match = re.search(r'mse:([\d.]+)', metrics_line)
        mae_match = re.search(r'mae:([\d.]+)', metrics_line)

        if mse_match and mae_match:
            results.append({
                'Dataset': dataset_name,
                'Model': model_type,
                'AttentionMode': attention_mode,
                'Variant': variant,
                'MSE': float(mse_match.group(1)),
                'MAE': float(mae_match.group(1)),
            })

    if results:
        df = pd.DataFrame(results)
        df = df.sort_values(['Dataset', 'Model', 'AttentionMode']).reset_index(drop=True)
        print("\n" + "=" * 90)
        print("RAW RESULTS")
        print("=" * 90)
        print(df)

        variant_order = ['SWT_original', 'SWT_dual', 'FFT_original', 'FFT_dual']
        pivot_df = df.pivot(index='Dataset', columns='Variant', values=['MSE', 'MAE'])
        pivot_df = pivot_df.reindex(columns=variant_order, level=1)

        print("\n" + "=" * 90)
        print("FINAL COMPARISON TABLE")
        print("=" * 90)
        print(pivot_df)
        print("=" * 90)

        df.to_csv(os.path.join(OUTPUT_DIR, 'raw_metrics.csv'), index=False)
        pivot_df.to_csv(os.path.join(OUTPUT_DIR, 'final_metrics.csv'))
        print(f"Saved raw results to: {os.path.join(OUTPUT_DIR, 'raw_metrics.csv')}")
        print(f"Saved final table to: {os.path.join(OUTPUT_DIR, 'final_metrics.csv')}")
    else:
        print('No valid metrics found in result file.')
else:
    print("Result file not found.")



RAW RESULTS
    Dataset         Model AttentionMode       Variant       MSE       MAE
0     ETTh1  SimpleTM_FFT          dual      FFT_dual  0.448829  0.444854
1     ETTh1  SimpleTM_FFT      original  FFT_original  0.457091  0.449469
2     ETTh1  SimpleTM_SWT          dual      SWT_dual  0.446452  0.443600
3     ETTh1  SimpleTM_SWT      original  SWT_original  0.451259  0.446162
4     ETTh2  SimpleTM_FFT          dual      FFT_dual  0.314468  0.362235
5     ETTh2  SimpleTM_FFT      original  FFT_original  0.313288  0.360546
6     ETTh2  SimpleTM_SWT          dual      SWT_dual  0.314802  0.362613
7     ETTh2  SimpleTM_SWT      original  SWT_original  0.312984  0.360527
8     ETTm1  SimpleTM_FFT          dual      FFT_dual  0.357032  0.385665
9     ETTm1  SimpleTM_FFT      original  FFT_original  0.363031  0.382873
10    ETTm1  SimpleTM_SWT          dual      SWT_dual  0.358250  0.384758
11    ETTm1  SimpleTM_SWT      original  SWT_original  0.349717  0.379085
12    ETTm2  SimpleTM_FFT

In [7]:
print("Packing logs, weights, plots, and results...")
shutil.make_archive("/kaggle/working/SimpleTM_DualAttention_All_Experiments", 'zip', OUTPUT_DIR)
print("\nDONE! Download 'SimpleTM_DualAttention_All_Experiments.zip' from the Kaggle Output pane.")
print("It contains checkpoints, plots, raw metrics, and the final comparison CSV.")


Packing logs, weights, plots, and results...

DONE! Download 'SimpleTM_DualAttention_All_Experiments.zip' from the Kaggle Output pane.
It contains checkpoints, plots, raw metrics, and the final comparison CSV.
